In [17]:
import sys
sys.path.append("../src")

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from machineguard.modeling import build_random_forest_model

In [18]:
DATA_PATH = "../data/ai4i2020.csv"
df = pd.read_csv(DATA_PATH)

df["temperature_difference"] = (
    df["Process temperature [K]"] - df["Air temperature [K]"]
)
df["power_proxy"] = (
    df["Torque [Nm]"] * df["Rotational speed [rpm]"]
)

In [19]:
feature_cols = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "temperature_difference",
    "power_proxy",
]

target_col = "Machine failure"

X = df[feature_cols]
y = df[target_col]

In [20]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

In [21]:
def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [22]:
categorical_features = ["Type"]
numeric_features = [col for col in feature_cols if col!="Type"]

In [23]:
linear_preprocessing = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", StandardScaler(), numeric_features)
    ]
)

In [24]:
logistic_model = Pipeline(
    steps=[
        ("preprocessing", linear_preprocessing),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
        ))
    ]
)

In [25]:
random_forest_model = build_random_forest_model(feature_cols)

In [26]:
tree_preprocessing = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

In [27]:
boosting_model = Pipeline(
    steps=[
        ("preprocessing", tree_preprocessing),
        ("classifier", HistGradientBoostingClassifier(
            random_state=42,
            class_weight="balanced",
        )),
    ]
)

In [28]:
logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
boosting_model.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
threshold = 0.3

logistic_prob = logistic_model.predict_proba(X_val)[:, 1]
rf_prob = random_forest_model.predict_proba(X_val)[:, 1]
boosting_prob = boosting_model.predict_proba(X_val)[:, 1]

logistic_pred = (logistic_prob >= threshold).astype(int)
rf_pred = (rf_prob >= threshold).astype(int)
boosting_pred = (boosting_prob >= threshold).astype(int)

In [30]:
logistic_metrics = calculate_metrics(y_val, logistic_pred)
rf_metrics = calculate_metrics(y_val, rf_pred)
boosting_metrics = calculate_metrics(y_val, boosting_pred)

pd.DataFrame(
    [logistic_metrics, rf_metrics, boosting_metrics],
    index=["Logistic Regression", "Random Forest", "HistGradientBoosting"]
)

,accuracy,precision,recall,f1
Logistic Regression,0.731,0.101695,0.882353,0.182371
Random Forest,0.989,0.896552,0.764706,0.825397
HistGradientBoosting,0.978,0.653846,0.750000,0.698630


In [31]:
print("Logistic Regression")
print(confusion_matrix(y_val, logistic_pred))

print("\nRandom Forest")
print(confusion_matrix(y_val, rf_pred))

print("\nHistGradientBoosting")
print(confusion_matrix(y_val, boosting_pred))

Logistic Regression
[[1402  530]
 [   8   60]]

Random Forest
[[1926    6]
 [  16   52]]

HistGradientBoosting
[[1905   27]
 [  17   51]]


- Random forest gives the best validation F1 score.
- Logistic regression has the highest recall, but it creates 530 false alarms.
- Random forest catech 52 failures with only 6 false alarms, the best balance.
- HistGradientBoosting performs better than logistic regression in precision, but it does not beat random forest on F1.